# Лабораторная работа №3
## Разработка запросов в SQLite
**База данных:** Автостоянка (Вариант 11б)

**Задачи:**
1. Создать базу данных и заполнить тестовыми данными
2. Разработать многотабличные запросы на выборку
3. Объединить запросы оператором UNION
4. Создать перекрёстный запрос (PIVOT)
5. Освоить работу с временными таблицами
6. Реализовать параметризованный запрос

In [59]:
import sqlite3

conn = sqlite3.connect('parking.db')
cursor = conn.cursor()

In [60]:
import pandas as pd

def show_cars():
    return pd.read_sql("SELECT * FROM Cars", conn)

def show_parking_access():
    return pd.read_sql("SELECT * FROM Parking_Access", conn)

def show_payments():
    return pd.read_sql("SELECT * FROM Payments", conn)

## 1. Создание таблиц

Схема БД «Автостоянка»:
- **Cars** — автомобили (гос. номер, тип, номер клиента)
- **Parking_Access** — допуск на автостоянку (гос. номер + дата = составной PK, разрешение)
- **Payments** — платежи за въезд (номер проезда, гос. номер, дата, время, тип)

In [61]:
def CreateTables():
    query = """
        CREATE TABLE IF NOT EXISTS Cars (
            License_Plate TEXT PRIMARY KEY,
            Car_Type      TEXT,
            Client_ID     INTEGER NOT NULL
        );

        CREATE TABLE IF NOT EXISTS Parking_Access (
            License_Plate TEXT NOT NULL,
            Work_Date     TEXT NOT NULL,
            Access_Permit INTEGER NOT NULL,
            PRIMARY KEY (License_Plate, Work_Date),
            FOREIGN KEY (License_Plate) REFERENCES Cars(License_Plate)
        );

        CREATE TABLE IF NOT EXISTS Payments (
            Entry_ID      INTEGER PRIMARY KEY,
            License_Plate TEXT NOT NULL,
            Work_Date     TEXT NOT NULL,
            Entry_Time    TEXT NOT NULL,
            Entry_Type    TEXT,
            FOREIGN KEY (License_Plate, Work_Date) REFERENCES Parking_Access(License_Plate, Work_Date)
        );
    """
    cursor.executescript(query)

In [62]:
CreateTables()


## 2. Заполнение данными

In [63]:
def InsertData():
    query = """
        INSERT OR IGNORE INTO Cars VALUES
            ('A123AA77',  'Sedan',      110),
            ('B555BB99',  'SUV',        102),
            ('C333CC199', 'SUV',        107),
            ('E777EE177', 'Sedan',      103),
            ('H888HH777', 'Motorcycle', 106),
            ('K001KK750', 'Truck',      104),
            ('M444MM97',  'SUV',        105),
            ('P999PP77',  'Truck',      108),
            ('T111TT99',  'Sedan',      109),
            ('X222XX50',  'Sedan',      101);
    """
    cursor.execute(query)

    query = """
        INSERT OR IGNORE INTO Parking_Access VALUES
            ('A123AA77',  '2025-08-27', 1),
            ('B555BB99',  '2025-06-02', 1),
            ('C333CC199', '2025-11-25', 0),
            ('E777EE177', '2025-10-15', 1),
            ('H888HH777', '2026-01-14', 1),
            ('K001KK750', '2025-12-17', 0),
            ('M444MM97',  '2025-11-19', 1),
            ('P999PP77',  '2025-10-22', 1),
            ('T111TT99',  '2025-09-24', 1),
            ('X222XX50',  '2025-10-21', 1);
    """
    cursor.execute(query)

    query = """
        INSERT OR IGNORE INTO Payments VALUES
            (1,  'A123AA77',  '2025-08-27', '08:30', 'Single'),
            (2,  'B555BB99',  '2025-06-02', '09:50', 'Subscription'),
            (3,  'E777EE177', '2025-10-15', '10:09', 'Single'),
            (4,  'H888HH777', '2026-01-14', '11:22', 'Subscription'),
            (5,  'M444MM97',  '2025-11-19', '08:25', 'Single'),
            (6,  'P999PP77',  '2025-10-22', '08:45', 'Subscription'),
            (7,  'T111TT99',  '2025-09-24', '13:06', 'Single'),
            (8,  'X222XX50',  '2025-10-21', '01:07', 'Subscription'),
            (9,  'A123AA77',  '2025-08-27', '18:04', 'Single'),
            (10, 'B555BB99',  '2025-06-02', '20:00', 'Single');
    """
    cursor.executescript(query)

InsertData()

## 3. Многотабличные запросы (SELECT)

Запросы с JOIN по двум и трём таблицам одновременно.

**Запрос 1** — все въезды с типом автомобиля (JOIN трёх таблиц)

**Запрос 2** — только автомобили с разрешённым допуском

In [64]:
def select_queries():
    print("--- 1. Все въезды с типом авто ---")
    return pd.read_sql("""
        SELECT 
            p.Entry_ID,
            p.License_Plate,
            c.Car_Type,
            pa.Work_Date,
            p.Entry_Time,
            p.Entry_Type
        FROM Payments p
        JOIN Parking_Access pa ON p.License_Plate = pa.License_Plate 
            AND p.Work_Date = pa.Work_Date
        JOIN Cars c ON c.License_Plate = p.License_Plate
        ORDER BY pa.Work_Date
    """, conn)

def select_queries_2():
    return pd.read_sql("""
        SELECT 
            c.License_Plate,
            c.Car_Type,
            c.Client_ID,
            pa.Work_Date,
            pa.Access_Permit
        FROM Cars c
        JOIN Parking_Access pa ON c.License_Plate = pa.License_Plate
        WHERE pa.Access_Permit = 1
    """, conn)

In [65]:
select_queries()

--- 1. Все въезды с типом авто ---


,Entry_ID,License_Plate,Car_Type,Work_Date,Entry_Time,Entry_Type
0,2,B555BB99,SUV,2025-06-02,09:50,Subscription
1,10,B555BB99,SUV,2025-06-02,20:00,Single
2,1,A123AA77,Sedan,2025-08-27,08:30,Single
3,9,A123AA77,Sedan,2025-08-27,18:04,Single
4,7,T111TT99,Sedan,2025-09-24,13:06,Single
5,3,E777EE177,Sedan,2025-10-15,10:09,Single
6,8,X222XX50,Sedan,2025-10-21,01:07,Subscription
7,6,P999PP77,Truck,2025-10-22,08:45,Subscription
8,5,M444MM97,SUV,2025-11-19,08:25,Single
9,4,H888HH777,Motorcycle,2026-01-14,11:22,Subscription


In [66]:
select_queries_2()

,License_Plate,Car_Type,Client_ID,Work_Date,Access_Permit
0,A123AA77,Sedan,110,2025-08-27,1
1,B555BB99,SUV,102,2025-06-02,1
2,E777EE177,Sedan,103,2025-10-15,1
3,H888HH777,Motorcycle,106,2026-01-14,1
4,M444MM97,SUV,105,2025-11-19,1
5,P999PP77,Truck,108,2025-10-22,1
6,T111TT99,Sedan,109,2025-09-24,1
7,X222XX50,Sedan,101,2025-10-21,1


## 4. Объединение запросов (UNION)

Оператор `UNION` объединяет результаты двух запросов с одинаковой структурой.

Здесь объединяем автомобили с допуском и без — добавляем колонку `Status` для различия.

In [67]:
def union_query():
    return pd.read_sql("""
        SELECT License_Plate, Car_Type, 'Has Access' AS Status
        FROM Cars
        JOIN Parking_Access USING (License_Plate)
        WHERE Access_Permit = 1

        UNION

        SELECT License_Plate, Car_Type, 'No Access' AS Status
        FROM Cars
        JOIN Parking_Access USING (License_Plate)
        WHERE Access_Permit = 0
    """, conn)

In [68]:
union_query()

,License_Plate,Car_Type,Status
0,A123AA77,Sedan,Has Access
1,B555BB99,SUV,Has Access
2,C333CC199,SUV,No Access
3,E777EE177,Sedan,Has Access
4,H888HH777,Motorcycle,Has Access
5,K001KK750,Truck,No Access
6,M444MM97,SUV,Has Access
7,P999PP77,Truck,Has Access
8,T111TT99,Sedan,Has Access
9,X222XX50,Sedan,Has Access


## 5. Перекрёстный запрос (PIVOT)

В SQLite `PIVOT` реализуется через `CASE WHEN + GROUP BY`.

Считаем количество въездов типа `Single` и `Subscription` по каждому месяцу.

In [69]:
def pivot_query():
    return pd.read_sql("""
        SELECT
            strftime('%Y-%m', Work_Date) AS Month,
            COUNT(CASE WHEN Entry_Type = 'Single' THEN 1 END) AS Single,
            COUNT(CASE WHEN Entry_Type = 'Subscription' THEN 1 END) AS Subscription
        FROM Payments
        GROUP BY Month
        ORDER BY Month
    """, conn)

In [70]:
pivot_query()

,Month,Single,Subscription
0,2025-06,1,1
1,2025-08,2,0
2,2025-09,1,0
3,2025-10,1,2
4,2025-11,1,0
5,2026-01,0,1


## 6. Временные таблицы

Временная таблица существует только в рамках текущей сессии и удаляется автоматически при закрытии соединения.

Шаги: создание → просмотр → изменение (добавление колонки) → удаление.

In [71]:
def temp_table():
    # Создание
    cursor.executescript("""
        CREATE TEMP TABLE IF NOT EXISTS Active_Cars AS
        SELECT c.License_Plate, c.Car_Type, pa.Work_Date
        FROM Cars c
        JOIN Parking_Access pa ON c.License_Plate = pa.License_Plate
        WHERE pa.Access_Permit = 1;
    """)
    
    # Просмотр
    display(pd.read_sql("SELECT * FROM Active_Cars", conn))
    
    # Изменение — добавим колонку
    cursor.execute("ALTER TABLE Active_Cars ADD COLUMN Note TEXT DEFAULT 'Active'")
    display(pd.read_sql("SELECT * FROM Active_Cars", conn))
    
    # Удаление
    cursor.execute("DROP TABLE Active_Cars")
    print("Temp table dropped")

In [72]:
temp_table()

,License_Plate,Car_Type,Work_Date
0,A123AA77,Sedan,2025-08-27
1,B555BB99,SUV,2025-06-02
2,E777EE177,Sedan,2025-10-15
3,H888HH777,Motorcycle,2026-01-14
4,M444MM97,SUV,2025-11-19
5,P999PP77,Truck,2025-10-22
6,T111TT99,Sedan,2025-09-24
7,X222XX50,Sedan,2025-10-21


,License_Plate,Car_Type,Work_Date,Note
0,A123AA77,Sedan,2025-08-27,Active
1,B555BB99,SUV,2025-06-02,Active
2,E777EE177,Sedan,2025-10-15,Active
3,H888HH777,Motorcycle,2026-01-14,Active
4,M444MM97,SUV,2025-11-19,Active
5,P999PP77,Truck,2025-10-22,Active
6,T111TT99,Sedan,2025-09-24,Active
7,X222XX50,Sedan,2025-10-21,Active


Temp table dropped


## 7. Параметризованный запрос

Параметр передаётся через `?` в SQLite — значение подставляется при вызове функции.

Фильтруем платежи по типу въезда: `Single` или `Subscription`.

In [73]:
def param_query(entry_type):
    return pd.read_sql("""
        SELECT 
            p.Entry_ID,
            p.License_Plate,
            c.Car_Type,
            p.Entry_Time,
            p.Entry_Type
        FROM Payments p
        JOIN Cars c ON p.License_Plate = c.License_Plate
        WHERE p.Entry_Type = ?
    """, conn, params=(entry_type,))

In [74]:
param_query('Single')

,Entry_ID,License_Plate,Car_Type,Entry_Time,Entry_Type
0,1,A123AA77,Sedan,08:30,Single
1,3,E777EE177,Sedan,10:09,Single
2,5,M444MM97,SUV,08:25,Single
3,7,T111TT99,Sedan,13:06,Single
4,9,A123AA77,Sedan,18:04,Single
5,10,B555BB99,SUV,20:00,Single


In [75]:

param_query('Subscription')

,Entry_ID,License_Plate,Car_Type,Entry_Time,Entry_Type
0,2,B555BB99,SUV,09:50,Subscription
1,4,H888HH777,Motorcycle,11:22,Subscription
2,6,P999PP77,Truck,08:45,Subscription
3,8,X222XX50,Sedan,01:07,Subscription
